In [1]:
import pandas as pd
import numpy as np
from datetime import datetime

class FinanceControlTower:

    def __init__(self, invoices_df, payments_df, operating_expenses_monthly):
        """
        invoices_df columns required:
        invoice_id, customer_id, invoice_date,
        service_start, service_end,
        invoice_amount, payment_status

        payments_df columns required:
        payment_id, invoice_id,
        payment_date, payment_amount
        """

        self.invoices = invoices_df.copy()
        self.payments = payments_df.copy()
        self.opex_monthly = operating_expenses_monthly

        self.revenue_schedule = None

    # ------------------------------------------------------
    # 1️⃣ Revenue Allocation Engine (IFRS 15 aligned)
    # ------------------------------------------------------
    def generate_revenue_schedule(self):

        revenue_rows = []

        for _, row in self.invoices.iterrows():

            start = pd.to_datetime(row["service_start"])
            end = pd.to_datetime(row["service_end"])
            amount = row["invoice_amount"]

            service_days = (end - start).days + 1
            daily_rate = amount / service_days

            current = start

            while current <= end:
                month_start = current.replace(day=1)
                month_end = (month_start + pd.offsets.MonthEnd(0))

                period_start = max(start, month_start)
                period_end = min(end, month_end)

                days_in_period = (period_end - period_start).days + 1
                revenue = days_in_period * daily_rate

                revenue_rows.append({
                    "invoice_id": row["invoice_id"],
                    "customer_id": row["customer_id"],
                    "revenue_month": month_start.strftime("%Y-%m"),
                    "recognized_revenue": revenue
                })

                current = month_end + pd.Timedelta(days=1)

        self.revenue_schedule = pd.DataFrame(revenue_rows)
        return self.revenue_schedule

    # ------------------------------------------------------
    # 2️⃣ Deferred Revenue Calculation
    # ------------------------------------------------------
    def calculate_deferred_revenue(self, as_of_date):

        as_of_date = pd.to_datetime(as_of_date)

        recognized = self.revenue_schedule.copy()
        recognized["revenue_month_date"] = pd.to_datetime(recognized["revenue_month"] + "-01")

        recognized_to_date = recognized[
            recognized["revenue_month_date"] <= as_of_date
        ].groupby("invoice_id")["recognized_revenue"].sum().reset_index()

        merged = self.invoices.merge(recognized_to_date, on="invoice_id", how="left")
        merged["recognized_revenue"] = merged["recognized_revenue"].fillna(0)

        merged["deferred_revenue"] = (
            merged["invoice_amount"] - merged["recognized_revenue"]
        )

        return merged[["invoice_id", "invoice_amount",
                       "recognized_revenue", "deferred_revenue"]]

    # ------------------------------------------------------
    # 3️⃣ AR Aging Model
    # ------------------------------------------------------
    def ar_aging(self, as_of_date):

        as_of_date = pd.to_datetime(as_of_date)

        ar = self.invoices.copy()
        ar["invoice_date"] = pd.to_datetime(ar["invoice_date"])

        ar["days_outstanding"] = (as_of_date - ar["invoice_date"]).dt.days

        conditions = [
            ar["days_outstanding"] <= 30,
            ar["days_outstanding"].between(31, 60),
            ar["days_outstanding"].between(61, 90),
            ar["days_outstanding"] > 90
        ]

        buckets = ["0-30", "31-60", "61-90", "90+"]

        ar["aging_bucket"] = np.select(conditions, buckets)

        aging_summary = ar.groupby("aging_bucket")["invoice_amount"].sum().reset_index()

        return aging_summary

    # ------------------------------------------------------
    # 4️⃣ Cash Runway Model
    # ------------------------------------------------------
    def cash_runway(self, current_cash):

        total_receivables = self.invoices["invoice_amount"].sum()
        total_payments = self.payments["payment_amount"].sum()

        net_receivables = total_receivables - total_payments

        monthly_burn = self.opex_monthly
        runway_months = (current_cash + net_receivables) / monthly_burn

        return {
            "Current Cash": current_cash,
            "Net Receivables": net_receivables,
            "Monthly Burn": monthly_burn,
            "Runway (Months)": round(runway_months, 2)
        }

    # ------------------------------------------------------
    # 5️⃣ CFO KPI Summary
    # ------------------------------------------------------
    def kpi_summary(self):

        total_revenue = self.revenue_schedule["recognized_revenue"].sum()
        total_invoiced = self.invoices["invoice_amount"].sum()
        total_collected = self.payments["payment_amount"].sum()

        collection_rate = total_collected / total_invoiced

        return {
            "Total Recognized Revenue": round(total_revenue, 2),
            "Total Invoiced": round(total_invoiced, 2),
            "Total Collected": round(total_collected, 2),
            "Collection Rate": round(collection_rate * 100, 2)
        }

In [2]:
# Sample Data
invoices = pd.DataFrame({
    "invoice_id": [1],
    "customer_id": ["C001"],
    "invoice_date": ["2025-01-01"],
    "service_start": ["2025-01-01"],
    "service_end": ["2025-12-31"],
    "invoice_amount": [12000],
    "payment_status": ["Open"]
})

payments = pd.DataFrame({
    "payment_id": [1],
    "invoice_id": [1],
    "payment_date": ["2025-02-01"],
    "payment_amount": [2000]
})

tower = FinanceControlTower(invoices, payments, operating_expenses_monthly=5000)

tower.generate_revenue_schedule()
print(tower.kpi_summary())
print(tower.ar_aging("2025-03-31"))
print(tower.cash_runway(current_cash=30000))

{'Total Recognized Revenue': np.float64(12000.0), 'Total Invoiced': np.int64(12000), 'Total Collected': np.int64(2000), 'Collection Rate': np.float64(16.67)}


TypeError: Choicelist and default value do not have a common dtype: The DType <class 'numpy.dtypes._PyLongDType'> could not be promoted by <class 'numpy.dtypes.StrDType'>. This means that no common DType exists for the given inputs. For example they cannot be stored in a single array unless the dtype is `object`. The full list of DTypes is: (<class 'numpy.dtypes.StrDType'>, <class 'numpy.dtypes.StrDType'>, <class 'numpy.dtypes.StrDType'>, <class 'numpy.dtypes.StrDType'>, <class 'numpy.dtypes._PyLongDType'>)